# Bjerknes feedback changes over time

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Specify kwargs

In [ ]:
## MONTH FOR PLOTTING
# SEL_MONTH = lambda x: x.sel(month=slice(10, 12)).mean("month")
# SEL_MONTH = lambda x: x.sel(month=slice(1,3)).mean("month")
SEL_MONTH = lambda x: x.mean("month")
MONTH0 = 10

## WHETHER TO COMPUTE BY MEMBER
BY_MEMBER = True

if BY_MEMBER:
    DIMS = ["time"]
else:
    DIMS = ["time", "member"]

## Load pre-computed regression coefficients

In [ ]:
## SPECIFY t_variable
T_VAR = "T_3"

## path where data is located
save_fp = pathlib.Path(DATA_FP, "bjerknes_zero_median", f"coefs_xvars={T_VAR}-h_w")
load = lambda n: xr.open_dataset(save_fp / f"{n}.nc")
coefs = {n: load(n) for n in ["all", "pos", "neg"]}

## path where data is located
save_fp = pathlib.Path(DATA_FP, "bjerknes_zero_median", f"coefs_xvars={T_VAR}")
load = lambda n: xr.open_dataset(save_fp / f"{n}.nc").squeeze(drop=True)
coefs_T = {n: load(n) for n in ["all", "pos", "neg"]}

## Funcs

### Misc

In [ ]:
def window(x, subtract_median=True):
    """get data in windows"""
    x_windowed = src.utils.get_windowed(x, stride=120)

    ## handle median case
    if subtract_median:
        median = x_windowed.groupby("time.month").median(["time", "member"])
        x_windowed = x_windowed.groupby("time.month") - median

    return x_windowed


def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


def regress_over_time(
    data, x_vars, y_vars, dims=["time", "member"], subtract_median=True
):
    """regression over time"""

    ## get windowed data
    if "year" in data.dims:
        data_ = data
    else:
        data_ = window(data, subtract_median=subtract_median)

    ## empty list to hold coefficients
    coefs = []

    ## shared args
    kwargs = dict(x_vars=x_vars, y_vars=y_vars, dims=dims)

    ## loop thru years
    for year in tqdm.tqdm(data_.year):

        ## get grouped data
        data_y = data_.sel(year=year).groupby("time.month")

        ## do regression
        coefs.append(data_y.map(src.utils.regress_xr_proj, **kwargs))

    return xr.concat(coefs, dim=data_.year)


def regress_wrapper(data, x_vars, y_var, y_fn, dims=["time", "member"]):
    """regression over time"""

    ## prep data
    y_data = src.utils.reconstruct_wrapper(data[[f"{y_var}", f"{y_var}_comp"]], fn=y_fn)

    ## subset for data
    data_ = xr.merge([data[x_vars], y_data])

    return regress_over_time(data_, x_vars=x_vars, y_vars=[y_var], dims=dims)


def frac_change(x):
    """fractional change"""
    return x / x.isel(year=0) - 1


def check_dims(x):
    """make sure dimensions are ok before averaging"""
    ## check if latitude is in ssh
    if "latitude" in x.dims:
        x_ = copy.deepcopy(x)
    else:
        x_ = x.expand_dims("latitude")

    ## check if z_t is in ssh
    if "z_t" in x.dims:
        x_ = copy.deepcopy(x_)
    else:
        x_ = x_.expand_dims("z_t")

    return x_


def get_ddt(data):
    """differentiate with respect to time"""
    data_ = copy.deepcopy(data)
    data_ = data_.assign_coords({"t_idx": ("time", np.arange(len(data_.time)))})
    data_ = data_.swap_dims({"time": "t_idx"})
    return data_.differentiate("t_idx").swap_dims({"t_idx": "time"})

### Plotting

In [ ]:
def make_scatter_ax(ax, anom_, xvar, yvar, month, label, by_season=True):
    """scatter plot of data for given month"""

    ## prep func
    if by_season:
        get_season = lambda x: src.utils.sel_month(
            x.resample({"time": "QS-JAN"}).mean(), month
        )

    else:
        get_season = lambda x: src.utils.sel_month(x, month)

    prep = lambda x: get_season(x).transpose("time", "member")

    ## get plot data
    plot_data = (prep(anom_[xvar]), prep(anom_[yvar]))

    ## get stats
    corr = xr.corr(*plot_data)
    cov = xr.cov(*plot_data)
    m = cov / plot_data[0].var()

    ## plot data
    ax.scatter(*plot_data, s=3, label=f"m = {m.item():.1e}\nr = {corr.item():.2f}")
    ax.set_title(f"{label}")

    ## formatting
    ax_kwargs = dict(ls="--", c="gray", lw=0.5)
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    ax.legend(prop=dict(size=10))

    return ax


def make_scatter(anom_, xvar, yvar, month, by_season=True):
    """scatter plot of data for given month"""

    fig, axs = plt.subplots(1, 4, figsize=(11, 2.5), layout="constrained")

    for ax, t_idx, label in zip(
        axs,
        [["1850", "1879"], ["1995", "2024"], ["2035", "2064"], ["2071", "2100"]],
        ["1865", "2010", "2050", "2085"],
    ):

        ## helper func
        prep = lambda x: x.sel(time=slice(*t_idx))

        ## scatter plot of data
        ax = make_scatter_ax(
            ax,
            prep(anom_),
            xvar=xvar,
            yvar=yvar,
            month=month,
            by_season=by_season,
            label=label,
        )

    ## format/scale axes
    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_timeseries(coefs, sel_fn=lambda x: x):
    """plot timeseries comparison"""

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5))

    ## loop thru pos and negative
    for i, (name, color) in enumerate(zip(["pos", "neg"], ["r", "b"])):

        ## plot median and bounds
        for q, lw in zip([0.5, 0.1, 0.9], [2, 0.6, 0.6]):

            ## plot neutral and pos/or neg
            for name_, color_ in zip(["all", name], ["k", color]):

                ## finally, plot data
                axs[i].plot(
                    coefs.year,
                    sel_fn(coefs)[name_].quantile(q=q, dim="member"),
                    c=color_,
                    lw=lw,
                )

            ## plot on shared axs
            axs[2].plot(
                coefs.year,
                sel_fn(coefs)[name].quantile(q=q, dim="member"),
                c=color,
                lw=lw,
            )

    ## formatting
    for ax in axs:
        ax.set_xticks([1870, 2010, 2080])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axvline(2010, **ax_kwargs)
        ax.axhline(0, **ax_kwargs)
    src.utils.set_lims(axs)

    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_zonal_structure(coefs, sel_fn=lambda x: x):
    """plot zonal structure of coefficients over time"""

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

    for ax, y in zip(axs, [1865, 2010, 2050, 2085]):

        ## get data for year
        ax.set_title(y)
        coefs_y = sel_fn(coefs).sel(year=y, method="nearest")

        ## select data
        for n, color in zip(["all", "pos", "neg"], ["k", "r", "b"]):

            ax.plot(coefs.longitude, coefs_y[n].mean("member"), c=color)

    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    for ax in axs:
        ax.set_xticks([140, 190, 240, 280])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axhline(0, **ax_kwargs)

    return fig, axs


def add_vticks(axs, xticks, xlines=None):
    """add vertical lines to axs"""

    ## specify line style
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)

    ## loop thru axs
    for ax in axs:
        ax.set_xticks(xticks)
        if xlines is not None:
            for x0 in xlines:
                ax.axvline(x0, **ax_kwargs)

    return


def plot_vertical_structure(coefs, sel_fn):
    """plot vertical structure of coefficients over time"""

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

    for ax, y in zip(axs, [1865, 2010, 2050, 2085]):

        ## get data for year
        ax.set_title(y)
        coefs_y = sel_fn(coefs).sel(year=y, method="nearest")

        ## select data
        for n, color in zip(["all", "pos", "neg"], ["k", "r", "b"]):

            ax.plot(coefs_y[n].mean("member"), coefs.z_t, c=color)

    for ax in axs:
        ax.set_ylim([150, 5])

        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axhline(50, **ax_kwargs)
        ax.axhline(80, **ax_kwargs)
        ax.axvline(0, **ax_kwargs)

    src.utils.set_lims(axs)
    axs[0].set_yticks([0, 50, 80])
    for ax in axs[1:]:
        ax.set_yticks([])
    return fig, axs

## Load data

#### $T$, $h$

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True)

### Spatial data

#### most data

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

#### "wide" subsurface data

In [ ]:
## should we use "wide" data?
USE_WIDE = True

## load spatial data
forced_wide, anom_wide = load_consolidated_wide()

if USE_WIDE:

    for v in list(forced_wide):
        forced[v] = forced_wide[v]
        anom[v] = anom_wide[v]

#### max grad thermocline

In [ ]:
h_mg_forced, h_mg = src.utils.load_h_data(max_grad=True)

### scaling for mean thermocline depth

#### Mean thermocline depth

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale_v2.nc"),
)

### Compute OHC

In [ ]:
## should we use mixed layer T?
# use_T_ml = True
use_T_ml = False

## specify subsetting funcs
LATS = dict(latitude=slice(-5, 5))
LATS_H = dict(latitude=slice(-5, 5))
LONS_E = dict(longitude=slice(210, 270))
# LONS_W = dict(longitude=slice(120, 160))
LONS_W = dict(longitude=slice(120, 180))
LONS_TAU = dict(longitude=slice(150, 230))

Set funcs

In [ ]:
## helper func to select and avg
def sel_helper(x, lats, lons):
    """helper func to avg over lats/lons"""

    ## first, average over lons
    x_avg = x.sel(lons).mean("longitude")

    if "latitude" in x_avg.dims:
        x_avg = x_avg.sel(lats).mean("latitude")

    return x_avg


## specify functions
TAU_FN = lambda x: sel_helper(x, LATS, LONS_TAU)
TAU_FN_3 = lambda x: sel_helper(x, LATS, LONS_E)
He_FN = lambda x: sel_helper(x, LATS_H, LONS_E)
Hw_FN = lambda x: sel_helper(x, LATS_H, LONS_W)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)


## specify entrainment / ML averages
LON_AVG = lambda x: x.sel(longitude=slice(210, 270)).mean("longitude")
LON_AVG_34 = lambda x: x.sel(longitude=slice(190, 240)).mean("longitude")
ENT_AVG = lambda x: x.sel(z_t=slice(50, 80)).mean("z_t")
ML_AVG = lambda x: x.sel(z_t=slice(None, 50)).mean("z_t")

## get T3 volume avg
T3_ENT_AVG = lambda x: ENT_AVG(LON_AVG(x))
T3_ML_AVG = lambda x: ML_AVG(LON_AVG(x))
T34_ML_AVG = lambda x: ML_AVG(LON_AVG_34(x))

if use_T_ml:
    anom["T_3"] = src.utils.reconstruct_wrapper(
        anom[["T", "T_comp"]],
        fn=T3_ML_AVG,
    )["T"]

In [ ]:
## should we use OHC?
USE_OHC = True

lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")

if USE_OHC:

    ## compute ohc
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W) / 300,
    )["T"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_E) / 300,
    )["T"]

else:
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_W)),
    )["ssh"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_E)),
    )["ssh"]

## handle rectification

In [ ]:
## get variables to check
rect_vars = ["u", "w", "T", "sst", "taux", "nhf"]

## get total
total = forced[rect_vars] + anom[rect_vars]

## window it
total = window(total, subtract_median=False).groupby("time.month").mean()

## add sigma T info
total["sigma_T"] = window(Th["T_34"]).groupby("time.month").std()

## estimate rect. coefficients
rect_coefs = src.utils.estimate_rect_bymonth(total, xvar="sigma_T", yvars=rect_vars)

## get climatology
clim = total.mean("member")
clim_norect = rect_coefs.sel(degree=0)

## Analysis

### T vs. ddt(SST)$

#### Load coefficients

In [ ]:
## specify which longitudes to reduce over
if T_VAR == "T_34":
    ddt_LON_RANGE = [190, 240]

else:
    ddt_LON_RANGE = [210, 270]

## specify which variable to use ("T" or "sst")
YVAR = "sst"

## function to select data
sel_var = lambda x: x[f"ddt_{YVAR}"].sel(j=T_VAR)

## load coefs
R_proj = 12 * xr.merge(sel_var(c).rename(n) for n, c in coefs.items())

#### Plot spatial structure

reconstruct spatial strucure

In [ ]:
def reduce_yz(x):

    if "z_t" not in x.dims:
        x = x.expand_dims("z_t")

    if "latitude" not in x.dims:
        x = x.expand_dims("latitude")

    idx = dict(z_t=slice(None, 50), latitude=slice(-5, 5))

    return x.sel(idx).mean(["z_t", "latitude"])


def reduce_x(x):

    ## get averaging dims
    idx = dict(longitude=slice(*ddt_LON_RANGE))

    return x.sel(idx).mean("longitude")


def recon_struct(coefs, v="T"):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename(v), anom[f"{v}_comp"]]),
        fn=reduce_yz,
    )

    return coefs_struct[v]


def recon_idx(coefs, v="T"):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename(v), anom[f"{v}_comp"]]),
        fn=lambda y: reduce_x(reduce_yz(y)),
    )

    return coefs_struct[v]

In [ ]:
## load coefs
R_struct = xr.merge(
    [recon_struct(R_proj[v], YVAR).rename(v) for v in ["all", "pos", "neg"]]
)

Plot

In [ ]:
fig, axs = plot_zonal_structure(R_struct, sel_fn=SEL_MONTH)
for ax in axs:
    ax.set_xlim([140, 280])

add_vticks(axs, xticks=[140, 190, 240, 280], xlines=[190, 240])
plt.show()

#### Timeseries

In [ ]:
## get index
R = reduce_x(R_struct)

fig, axs = plot_timeseries(coefs=R, sel_fn=SEL_MONTH)
plt.show()

#### Scatter plot

In [ ]:
scatter_data = xr.merge(
    [
        get_ddt(recon_idx(anom[YVAR], YVAR)).rename(f"ddt_{YVAR}"),
        anom[["T_3", "T_34", "h_w"]],
    ]
)

for xvar in [T_VAR, "h_w"]:
    fig, axs = make_scatter(scatter_data, xvar=xvar, yvar=f"ddt_{YVAR}", month=MONTH0)
    plt.show()

### Compare to RO

#### Get data

In [ ]:
## get data (remove median as well)
x = xr.merge([Th[T_VAR], anom["h_w"]])
# x /= x.std()
x = window(x, subtract_median=True)
y = get_ddt(x).rename({f"{v}":f"ddt_{v}" for v in list(x)})
z = xr.merge([x,y])

#### Version 1: piecewise linear

In [ ]:
## do regression
# z_pos = z.where(z[T_VAR]>0)

kwargs = dict(
    func=src.utils.regress_xr_proj, 
    x_vars=list(x), 
    y_vars=list(y), 
    dims=["time", "member"]
)

regress_by_month = lambda x : 12 * x.groupby("time.month").map(**kwargs)

## find pos/neg
is_pos_ = z[T_VAR]>0
is_neg_ = z[T_VAR]<0

## do regression
coefs2_all = regress_by_month(z)
coefs2_pos = regress_by_month(z.where(is_pos_))
coefs2_neg = regress_by_month(z.where(is_neg_))
coefs2 = [coefs2_all, coefs2_pos, coefs2_neg]

## get R, epsilon
get_R = lambda x : x["ddt_T_3"].sel(j="T_3")
get_eps = lambda x: -x["ddt_h_w"].sel(j="h_w")
R2 = xr.merge(
    [get_R(c).rename(n) for c,n in zip(coefs2, ["all","pos","neg"])]
)

Check reconstruction matches

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(8.5,2.5), layout="constrained")
for ax, n in zip(axs, list(R)):
    ax.plot(R2.year, R2[n].mean("month"))
    ax.plot(R.year, R[n].mean(["month","member"]), ls="--")
    ax.axhline(0, ls="--", c="k", lw=.8)
    ax.set_title(n)

src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])

add_vticks(axs, xticks=[1870,2010,2090], xlines=[2010,2090])
plt.show()

#### version 2: cubic

In [ ]:
import warnings

def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits

In [ ]:
## parameters for fitting
MODEL = src.XRO.XRO(ncycle=12, ac_order=4, is_forward=True)
fit_kwargs = dict(ac_mask_idx=None, maskNT=["T2", "T3"], maskNH=[])

## get fits
fits = get_fits_over_time(
    x,
    model=MODEL,
    by_member=False,
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

In [ ]:
## get coefficients
coefs_nl = fits["NROT_Lac"].sel(nro_form=["T2", "T3"])
R3 = fits["Lac"].isel(ranky=0, rankx=0).expand_dims(nro_form=["T"])
eps3 = -fits["Lac"].isel(ranky=1, rankx=1)
coefs3 = xr.concat([R3, coefs_nl], dim="nro_form").rename({"nro_form": "form"})

## get coefficient array
# T0 = 1.5
sigma = x["T_3"].std().values.item()
w = sigma * np.linspace(-3, 3)
W = np.stack([w**i for i in range(3)], axis=1)
W = xr.DataArray(W, coords=dict(T=w, form=coefs3.form), dims=["T", "form"])

## reconstruct nonlinear R
R_nl = (coefs3 * W).sum("form")
R_nl_pos = R_nl.where(R_nl["T"]>0).mean("T")
R_nl_neg = R_nl.where(R_nl["T"]<0).mean("T")


# T0 = sigma * 1.5
T0 = 1.5
R_nl_pos2 = R_nl.sel(T=T0, method="nearest")
R_nl_neg2 = R_nl.sel(T=-T0, method="nearest")

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(8.5,2.5), layout="constrained")
for ax, n in zip(axs, list(R)):
    ax.plot(R2.year, R2[n].mean("month"))
    ax.plot(R.year, R[n].mean(["month","member"]), ls="--")
    ax.axhline(0, ls="--", c="k", lw=.8)
    ax.set_title(n)

axs[0].plot(params.year, params["R"].mean("cycle"), label=r"$R_0$")
axs[0].plot(params.year, R_nl.mean(["cycle","T"]), label=r"avg($R$)")


axs[1].plot(params.year, R_nl_pos.mean("cycle"), c="gray")
axs[1].plot(params.year, R_nl_pos2.mean("cycle"),  c="gray", ls="--")

axs[2].plot(params.year, R_nl_neg.mean("cycle"),  c="gray")
axs[2].plot(params.year, R_nl_neg2.mean("cycle"),  c="gray", ls="--")

axs[0].legend(prop=dict(size=8))
src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])

add_vticks(axs, xticks=[1870,2010,2090], xlines=[2010,2050, 2090])
plt.show()

\begin{align}
    R &= R_0 + R_1T + R_2 T^2
\end{align}

In [ ]:
sel = lambda x : x.mean('cycle')#.differentiate("year")
fig,ax = plt.subplots(figsize=(3,2.5))
ax.plot(params.year, .2*sel(params["R"]), c="k")
ax.plot(coefs_nl.year, sel(coefs_nl.sel(nro_form="T2")))
ax.plot(coefs_nl.year, sel(coefs_nl.sel(nro_form="T3")))
ax.axhline(0, ls="--", lw=.8, c="gray")
ax.axvline(2050, ls="--", lw=.8, c="gray")
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(3,2.5), layout="constrained")

ax.plot(params.year, R_nl.mean(["T","cycle"]), c="k")
ax.plot(params.year, R_nl_pos.mean("cycle"), c="r")
ax.plot(params.year, R_nl_neg.mean("cycle"),  c="b")

ax.plot(params.year, params["R"].mean("cycle"), c="k", ls="--")
ax.plot(params.year, R_nl_pos2.mean("cycle"), c="r", ls="--")
ax.plot(params.year, R_nl_neg2.mean("cycle"),  c="b", ls="--")

add_vticks([ax], xticks=[1870,2010,2090], xlines=[2010,2050])
plt.show()

color by magnitude of anomaly...

### Damping

#### Load data

In [ ]:
## get scaling factor to convert to damping rate (units of K/mo)
sec_per_mo = 8.64e4 * 30
rho = 1.02e3
Cp = 4.2e3
H0 = 50
scale_fac = sec_per_mo / (rho * Cp * H0)

## helper func to get data
prep = lambda x: -12 * x["nhf"] * scale_fac

## load coefs
alpha_proj = xr.merge(prep(c).rename(n) for n, c in coefs_T.items())

#### structure

In [ ]:
def reduce_x(x):
    """Get Tsub, but don't average over depth yet"""

    return x.sel(longitude=slice(210, 270)).mean("longitude")


def reduce_y(x):
    """Get diff b/n Tsub and Tsurf, but don't average over longitude yet"""

    return x.sel(latitude=slice(-5, 5)).mean("latitude")


def recon_struct(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename("nhf"), anom["nhf_comp"]]),
        fn=reduce_y,
    )
    ## drop extra variables
    coefs_ = coefs_["nhf"].squeeze(drop=True)

    return coefs_


def recon_idx(coefs):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs["nhf"], anom[f"nhf_comp"]]),
        fn=lambda z: reduce_x(reduce_y(z)),
    )

    return coefs_struct["nhf"]

In [ ]:
## load coefs
alpha_struct = xr.merge(
    [recon_struct(alpha_proj[v]).rename(v) for v in ["all", "pos", "neg"]]
)

In [ ]:
fig, axs = plot_zonal_structure(coefs=alpha_struct, sel_fn=SEL_MONTH)
add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")
for ax in axs:
    ax.set_xlim([140, 280])
plt.show()

#### Timeseries

In [ ]:
alpha = reduce_x(alpha_struct)

fig, axs = plot_timeseries(coefs=alpha, sel_fn=SEL_MONTH)
plt.show()

#### Scatter

In [ ]:
scatter_data = xr.merge(
    [
        recon_idx(anom[["nhf"]]),
        anom[["T_3", "T_34"]],
    ]
)
fig, axs = make_scatter(scatter_data, xvar=T_VAR, yvar="nhf", month=10)
plt.show()

## custom implementation

### Load data

#### NHF in Niño 3 region

In [ ]:
## get scaling factor to convert to damping rate (units of K/mo)
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H0 = 50
SCALE_FAC = sec_per_yr / (rho * Cp * H0)

Q_3 = SCALE_FAC * src.utils.reconstruct_wrapper(
    anom[["nhf","nhf_comp"]], fn=src.utils.get_nino3,
).rename({"nhf":"Q_3"})

#### merge data

In [ ]:
## get data (remove median as well)
XY = xr.merge([Th[T_VAR], anom["h_w"], Q_3])
XY = window(XY, subtract_median=True)

## differentiate T/hvars
Y = get_ddt(XY[[T_VAR,"h_w"]])
Y = Y.rename({f"{v}":f"ddt_{v}" for v in list(Y)})

### Fit models

### Plot data

In [ ]:
sel_mo = lambda x : x.sel(time=slice(None,None,12))
fig,axs = plt.subplots(1,3,figsize=(9,3))

for ax, y in zip(axs, [1870, 2010, 2080]):

    sel = lambda x : sel_mo(x).sel(year=y)
    
    ax.scatter(
        sel(XY["T_3"]), sel(XY["Q_3"]), s=2,
    )
    ax_kwargs = dict(lw=.8, ls="--", c="gray")
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)

src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])
plt.show()

In [ ]:
## load spatial data
_, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

In [ ]:
## get coefficients
coefs_nl = fits["NROT_Lac"].sel(nro_form=["T2", "T3"])
R3 = fits["Lac"].isel(ranky=0, rankx=0).expand_dims(nro_form=["T"])
eps3 = -fits["Lac"].isel(ranky=1, rankx=1)
coefs3 = xr.concat([R3, coefs_nl], dim="nro_form").rename({"nro_form": "form"})

## get coefficient array
# T0 = 1.5
sigma = x["T_3"].std().values.item()
w = sigma * np.linspace(-3, 3)
W = np.stack([w**i for i in range(3)], axis=1)
W = xr.DataArray(W, coords=dict(T=w, form=coefs3.form), dims=["T", "form"])

## reconstruct nonlinear R
R_nl = (coefs3 * W).sum("form")
R_nl_pos = R_nl.where(R_nl["T"]>0).mean("T")
R_nl_neg = R_nl.where(R_nl["T"]<0).mean("T")


# T0 = sigma * 1.5
T0 = 1.5
R_nl_pos2 = R_nl.sel(T=T0, method="nearest")
R_nl_neg2 = R_nl.sel(T=-T0, method="nearest")